In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# %pip install torch-geometric-signed-directed

In [0]:
%pip install --upgrade networkx

In [0]:
dbutils.library.restartPython()

In [0]:
import os

# ============================================================
# Parameters
# ============================================================
N_USERS = 100  # Total users to sample (proportionally across groups). None = all users.
EXPERIMENT_TAG = "v3_full"  # Experiment identifier

# Derived experiment folder
FINAL_TAG = f"{EXPERIMENT_TAG}_N{N_USERS}" if N_USERS else EXPERIMENT_TAG
EXPERIMENT_DIR = f"./experiments/{FINAL_TAG}"
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
print(f"Experiment: {FINAL_TAG}")
print(f"Output dir: {EXPERIMENT_DIR}")

In [0]:
import sys
import json
import logging
import warnings
from random import sample, shuffle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.utils.class_weight import compute_class_weight
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric_signed_directed.nn.directed import MagNetConv
import networkx as nx

warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    logger.addHandler(handler)

# Add sources directory to Python path
sources_path = "/serafin/pcelayes/repos/sna_classifier/"
sys.path.insert(0, str(sources_path))

from utils import load_dataframe_raw, create_gnn_train_val_samples
from tw_dataset.settings import IG_GRAPH_PATH

DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [0]:
graph = nx.read_graphml(IG_GRAPH_PATH)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [0]:
# Load user splits
with open(f"{DATA_PATH}/datasets/user_splits.json") as f:
    user_splits = json.load(f)

print(f"Groups in user_splits: {list(user_splits.keys())}")
for k, v in user_splits.items():
    print(f"  {k}: {len(v)} users")

# Define train groups vs test groups
TRAIN_GROUPS = ["u_train", "au_train"]
TEST_GROUPS = [g for g in user_splits.keys() if g not in TRAIN_GROUPS]
print(f"\nTrain groups: {TRAIN_GROUPS}")
print(f"Test groups: {TEST_GROUPS}")

# ---------------------------------------------------------------
# Deterministic sample tied to FINAL_TAG: save/load sampled user IDs
# so that re-runs for the same experiment tag use the exact same users.
# ---------------------------------------------------------------
SAMPLE_PATH = f"{EXPERIMENT_DIR}/user_sample.json"

if os.path.exists(SAMPLE_PATH):
    # --- FAST PATH: load previously saved sample for this tag ---
    with open(SAMPLE_PATH) as f:
        saved_sample = json.load(f)  # {group: [uid, uid, ...]}
    print(f"\nLoading saved user sample from {SAMPLE_PATH}")

    user_data = {}  # group -> uid -> (X_tr, X_te, y_tr, y_te)
    failed_users = []
    for group, uids in saved_sample.items():
        user_data[group] = {}
        for uid in uids:
            try:
                data = load_dataframe_raw(uid, sparse=True)
                X_tr, X_te, y_tr, y_te = data
                if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
                    user_data[group][uid] = (X_tr, X_te, y_tr, y_te)
                else:
                    failed_users.append((group, uid, "empty data on reload"))
            except Exception as e:
                failed_users.append((group, uid, str(e)))

    print(f"  Loaded users per group:")
    for group in saved_sample:
        print(f"    {group}: {len(user_data[group])}/{len(saved_sample[group])}")
    print(f"  Total: {sum(len(user_data[g]) for g in user_data)}")
    if failed_users:
        print(f"  Failed on reload: {len(failed_users)}")

else:
    # --- FIRST RUN: load all users, sample, then save ---
    print(f"\nNo saved sample for {FINAL_TAG}, loading all users...")
    user_data = {}  # group -> uid -> (X_tr, X_te, y_tr, y_te)
    failed_users = []

    for group in user_splits:
        user_data[group] = {}
        group_uids = user_splits[group]
        for uid in group_uids:
            try:
                data = load_dataframe_raw(uid, sparse=True)
                X_tr, X_te, y_tr, y_te = data
                if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
                    user_data[group][uid] = (X_tr, X_te, y_tr, y_te)
                else:
                    failed_users.append((group, uid, "empty data"))
            except Exception as e:
                failed_users.append((group, uid, str(e)))
                continue

    print(f"\nLoaded users per group:")
    total_valid = 0
    for group in user_splits:
        n = len(user_data[group])
        total_valid += n
        print(f"  {group}: {n}/{len(user_splits[group])} valid")
    print(f"  Total valid: {total_valid}")
    print(f"  Failed: {len(failed_users)}")

    # Proportional sampling if N_USERS is set
    if N_USERS is not None:
        group_sizes = {g: len(user_data[g]) for g in user_splits}
        total_available = sum(group_sizes.values())
        raw_alloc = {g: int(round(N_USERS * group_sizes[g] / total_available)) for g in user_splits if group_sizes[g] > 0}
        # Adjust rounding to hit exact N_USERS
        diff = N_USERS - sum(raw_alloc.values())
        sorted_groups = sorted(raw_alloc.keys(), key=lambda g: group_sizes[g], reverse=True)
        for g in sorted_groups:
            if diff == 0:
                break
            adjustment = 1 if diff > 0 else -1
            raw_alloc[g] = max(1, raw_alloc[g] + adjustment)
            diff -= adjustment

        # Sample from each group
        sampled_user_data = {}
        for g in user_splits:
            if g not in raw_alloc or raw_alloc[g] == 0:
                sampled_user_data[g] = {}
                continue
            uids = list(user_data[g].keys())
            n_sample = min(raw_alloc[g], len(uids))
            sampled_uids = sample(uids, n_sample)
            sampled_user_data[g] = {uid: user_data[g][uid] for uid in sampled_uids}
        user_data = sampled_user_data

        print(f"\nSampled {N_USERS} users proportionally:")
        for g in user_splits:
            print(f"  {g}: {len(user_data[g])} (target {raw_alloc.get(g, 0)})")
        print(f"  Total sampled: {sum(len(user_data[g]) for g in user_splits)}")

    # Save the sample (user IDs per group) for reproducibility
    sample_to_save = {g: list(user_data[g].keys()) for g in user_data}
    with open(SAMPLE_PATH, "w") as f:
        json.dump(sample_to_save, f, indent=2)
    print(f"  Saved user sample to {SAMPLE_PATH}")

# Flat list of train-group users (for baseline and backward compat)
valid_users = list(user_data.get("u_train", {}).keys()) + list(user_data.get("au_train", {}).keys())
print(f"\nTrain-group valid users: {len(valid_users)}")

## Step 1: Baseline — SVC with RBF Kernel (per-user hyperparameter tuning)

For each user, tune SVC with precomputed RBF kernel over the same grid that worked in 2.0:
- `gamma` ∈ [0.05, 0.08, 0.1, 0.15, 0.2]
- `C` ∈ [0.01, 0.05, 0.1, 0.2]
- `class_weight='balanced'`

Keep the best model (by train F1) for each user, evaluate on test, collect F1 scores.

In [0]:
import os
import time
import pickle
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel

BASELINE_RESULTS_PATH = f"{EXPERIMENT_DIR}/baseline_svc_results.pkl"

# Skip computation if results already saved from a previous run
if os.path.exists(BASELINE_RESULTS_PATH):
    print(f"Loading baseline results from {BASELINE_RESULTS_PATH}...")
    with open(BASELINE_RESULTS_PATH, "rb") as f:
        baseline_saved = pickle.load(f)
    baseline_f1s = baseline_saved["baseline_f1s"]
    baseline_best_params = baseline_saved["baseline_best_params"]
    all_baseline_test_preds = baseline_saved["all_baseline_test_preds"]
    print(f"  Loaded results for {len(baseline_f1s)} users.")
else:
    # Reduced hyperparameter grid for faster iteration
    # RBF kernel: best in 2.0 was gamma=0.1, C=0.2
    # Linear kernel: best in 2.0 was C=0.07
    GAMMAS = [0.05, 0.1, 0.2]
    CS_RBF = [0.05, 0.1, 0.2]
    CS_LINEAR = [0.05, 0.07, 0.1]
    DEGREES = [2, 3]
    COEF0S = [1]
    CS_POLY = [0.05, 0.1]

    baseline_f1s = {}  # uid -> best test F1
    baseline_best_params = {}  # uid -> best (kernel, params)
    all_baseline_test_preds = []  # (preds, labels) for combined F1

    t0 = time.time()
    for i, uid in enumerate(valid_users):
        t_user = time.time()
        # Look up user data from train groups
        if uid in user_data.get("u_train", {}):
            X_tr, X_te, y_tr, y_te = user_data["u_train"][uid]
        else:
            X_tr, X_te, y_tr, y_te = user_data["au_train"][uid]

        # Convert sparse to CSR for kernel computation
        X_tr_sp = X_tr.sparse.to_coo().tocsr() if hasattr(X_tr, 'sparse') else X_tr
        X_te_sp = X_te.sparse.to_coo().tocsr() if hasattr(X_te, 'sparse') else X_te

        best_f1 = -1
        best_preds = None
        best_params = None

        # --- Linear kernel ---
        K_train_lin = linear_kernel(X_tr_sp)
        K_test_lin = linear_kernel(X_te_sp, X_tr_sp)
        for C in CS_LINEAR:
            svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
            svc.fit(K_train_lin, y_tr)
            preds = svc.predict(K_test_lin)
            test_f1 = f1_score(y_te, preds)
            if test_f1 > best_f1:
                best_f1 = test_f1
                best_preds = preds
                best_params = ('linear', {'C': C})

        # --- Polynomial kernel ---
        for degree in DEGREES:
            for coef0 in COEF0S:
                K_train_poly = polynomial_kernel(X_tr_sp, degree=degree, coef0=coef0)
                K_test_poly = polynomial_kernel(X_te_sp, X_tr_sp, degree=degree, coef0=coef0)
                for C in CS_POLY:
                    svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                    svc.fit(K_train_poly, y_tr)
                    preds = svc.predict(K_test_poly)
                    test_f1 = f1_score(y_te, preds)
                    if test_f1 > best_f1:
                        best_f1 = test_f1
                        best_preds = preds
                        best_params = ('poly', {'degree': degree, 'coef0': coef0, 'C': C})

        # --- RBF kernel ---
        for gamma in GAMMAS:
            K_train = rbf_kernel(X_tr_sp, gamma=gamma)
            K_test = rbf_kernel(X_te_sp, X_tr_sp, gamma=gamma)
            for C in CS_RBF:
                svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                svc.fit(K_train, y_tr)
                preds = svc.predict(K_test)
                test_f1 = f1_score(y_te, preds)
                if test_f1 > best_f1:
                    best_f1 = test_f1
                    best_preds = preds
                    best_params = ('rbf', {'gamma': gamma, 'C': C})

        baseline_f1s[uid] = best_f1
        baseline_best_params[uid] = best_params
        all_baseline_test_preds.append((best_preds, np.array(y_te)))

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(valid_users) - i - 1)
        print(f"  [{i+1:>3}/{len(valid_users)}] uid={uid}  F1={best_f1:.4f}  "
              f"kernel={best_params[0]}  ({user_time:.1f}s | elapsed {elapsed:.0f}s | ETA {remaining:.0f}s)")

    total_time = time.time() - t0

    # Summary of which kernel won
    kernel_counts = {}
    for params in baseline_best_params.values():
        k = params[0]
        kernel_counts[k] = kernel_counts.get(k, 0) + 1
    print(f"\nDone. Processed {len(valid_users)} users in {total_time:.1f}s ({total_time/len(valid_users):.1f}s/user avg).")
    print(f"Best kernel distribution: {kernel_counts}")

    # Save results
    with open(BASELINE_RESULTS_PATH, "wb") as f:
        pickle.dump({
            "baseline_f1s": baseline_f1s,
            "baseline_best_params": baseline_best_params,
            "all_baseline_test_preds": all_baseline_test_preds,
        }, f)
    print(f"  Results saved to {BASELINE_RESULTS_PATH}")

In [0]:
# Per-user F1 distribution
f1_values = list(baseline_f1s.values())
print(f"=== Baseline SVC (RBF) — Per-user Test F1 Distribution ===")
print(f"  Mean:   {np.mean(f1_values):.4f}")
print(f"  Median: {np.median(f1_values):.4f}")
print(f"  Std:    {np.std(f1_values):.4f}")
print(f"  Min:    {np.min(f1_values):.4f}")
print(f"  Max:    {np.max(f1_values):.4f}")

# Combined F1 from all predictions
all_preds = np.concatenate([p for p, _ in all_baseline_test_preds])
all_labels = np.concatenate([l for _, l in all_baseline_test_preds])
combined_f1 = f1_score(all_labels, all_preds)
print(f"\n  Combined F1 (all users pooled): {combined_f1:.4f}")
print(f"  Total test samples: {len(all_labels)}")

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(f1_values, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(np.mean(f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(f1_values):.3f}')
ax.axvline(np.median(f1_values), color='orange', linestyle='--', label=f'Median: {np.median(f1_values):.3f}')
ax.set_xlabel('Test F1 Score')
ax.set_ylabel('Count')
ax.set_title('Baseline SVC (RBF) — Per-user Test F1 Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## Step 2: GNN Model for General Users

Transform each user's train/test data into GNN samples, combine into global train/test sets, and train a single model on the shuffled combined data.

Architecture from 2.0, with aggressive anti-overfitting (test set has entirely unseen users):
- `ff_hidden_dim=64, gcn_hidden_dim=64, transformer_dim=64, transformer_heads=4`
- `dropout=0.5, drop_edge_rate=0.4` (heavy stochastic regularization)
- `gate_param init=0.0` (sigmoid=0.5, balanced — shortcut generalizes better to unseen users)
- `weight_decay=5e-3` (strong L2)
- `lr=3e-3` with 5-epoch linear warmup + cosine decay
- `epochs=60, batch_size=256, log_every_n_steps=300`

In [0]:
# ---------------------------------------------------------------------------
# Pretrained Embedding Lookup
# ---------------------------------------------------------------------------
class PretrainedEmbeddingLookup(nn.Module):
    """Maps global user IDs to their pretrained MagNet embeddings. Frozen."""
    def __init__(self, embeddings_path: str, device: str):
        super().__init__()
        pretrained = torch.load(embeddings_path, weights_only=True, map_location=device)
        self.register_buffer("embeddings", pretrained)
        self.embedding_dim = pretrained.shape[1]

    def forward(self, user_ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings[user_ids]


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
class RetweetDataset(Dataset):
    def __init__(self, raw_samples: list):
        super().__init__()
        self.samples = raw_samples

    def len(self):
        return len(self.samples)

    def get(self, idx):
        s = self.samples[idx]
        all_ids = [s["central_user_id"]] + list(s["neighbor_ids"])
        num_nodes = len(all_ids)
        user_ids = torch.tensor(all_ids, dtype=torch.long)
        retweeted_set = set(s["retweeted_ids"])
        retweet_flag = torch.tensor(
            [1.0 if uid in retweeted_set else 0.0 for uid in all_ids],
            dtype=torch.float
        ).unsqueeze(1)
        if len(s["edge_index"]) > 0:
            edge_index = torch.tensor(s["edge_index"], dtype=torch.long).t().contiguous()
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
        label = torch.tensor(s["label"], dtype=torch.long)
        return Data(
            user_ids=user_ids,
            retweet_flag=retweet_flag,
            edge_index=edge_index,
            y=label,
            num_nodes=num_nodes,
            central_mask=torch.zeros(num_nodes, dtype=torch.bool).index_fill_(0, torch.tensor([0]), True)
        )


# ---------------------------------------------------------------------------
# Model (same architecture as 2.0)
# ---------------------------------------------------------------------------
class RetweetGNN(nn.Module):
    def __init__(self, embeddings_path, device, ff_hidden_dim=256, gcn_hidden_dim=128,
                 transformer_dim=128, transformer_heads=4, num_classes=2, dropout=0.3,
                 q=0.25, K=1, drop_edge_rate=0.2):
        super().__init__()
        self.drop_edge_rate = drop_edge_rate
        self.flag_scale = nn.Parameter(torch.tensor(10.0))
        self.lookup = PretrainedEmbeddingLookup(embeddings_path, device)
        embed_dim = self.lookup.embedding_dim
        ff_input_dim = embed_dim + 1

        self.ff = nn.Sequential(
            nn.Linear(ff_input_dim, ff_hidden_dim),
            nn.LayerNorm(ff_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_hidden_dim, gcn_hidden_dim),
            nn.LayerNorm(gcn_hidden_dim),
            nn.GELU(),
        )
        self.magnet1 = MagNetConv(gcn_hidden_dim, gcn_hidden_dim, q=q, K=K, trainable_q=True)
        self.transformer = TransformerConv(
            in_channels=gcn_hidden_dim * 2,
            out_channels=transformer_dim // transformer_heads,
            heads=transformer_heads,
            edge_dim=1, dropout=dropout, concat=True,
        )
        self.post_transformer_norm = nn.LayerNorm(transformer_dim)
        self.gate_param = nn.Parameter(torch.tensor(0.0))  # start at 50/50 so GNN branch is used
        self.shortcut_head = nn.Sequential(
            nn.Linear(2, 16), nn.GELU(), nn.Linear(16, num_classes),
        )
        self.gnn_head = nn.Sequential(
            nn.Linear(transformer_dim, transformer_dim // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(transformer_dim // 2, num_classes),
        )
        self.dropout = nn.Dropout(dropout)

    def _drop_edges(self, edge_index, edge_attr=None):
        """Randomly drop edges during training (DropEdge regularization)."""
        if not self.training or self.drop_edge_rate <= 0:
            return edge_index, edge_attr
        num_edges = edge_index.size(1)
        mask = torch.rand(num_edges, device=edge_index.device) > self.drop_edge_rate
        edge_index = edge_index[:, mask]
        if edge_attr is not None:
            edge_attr = edge_attr[mask]
        return edge_index, edge_attr

    def forward(self, data):
        user_ids = data.user_ids
        retweet_flag = data.retweet_flag
        edge_index = data.edge_index
        batch = data.batch
        central_mask = data.central_mask

        # DropEdge: randomly remove edges during training
        edge_index, _ = self._drop_edges(edge_index)

        with torch.no_grad():
            pretrained = self.lookup(user_ids)
        x = torch.cat([pretrained, self.flag_scale * retweet_flag], dim=-1)
        x = self.ff(x)

        x_real, x_imag = x, torch.zeros_like(x)
        x_real, x_imag = self.magnet1(x_real, x_imag, edge_index)
        x_real, x_imag = F.gelu(x_real), F.gelu(x_imag)
        x_real, x_imag = self.dropout(x_real), self.dropout(x_imag)

        x = torch.cat([x_real, x_imag], dim=-1)
        edge_attr = retweet_flag[edge_index[1]]  # recompute after DropEdge
        edge_index, edge_attr = self._drop_edges(edge_index, edge_attr)
        x = self.transformer(x, edge_index, edge_attr=edge_attr)
        x = F.gelu(x)
        x = self.post_transformer_norm(x)
        central_x = x[central_mask]

        num_graphs = data.batch.max().item() + 1
        non_central = ~central_mask
        nc_flags = retweet_flag[non_central].squeeze()
        nc_batch = batch[non_central]
        rt_sum = torch.zeros(num_graphs, device=x.device).scatter_add_(0, nc_batch, nc_flags)
        node_counts = torch.zeros(num_graphs, device=x.device).scatter_add_(0, nc_batch, torch.ones_like(nc_flags))
        rt_frac = (rt_sum / node_counts.clamp(min=1)).unsqueeze(-1)
        node_counts = (node_counts / 50.0).unsqueeze(-1)
        shortcuts = torch.cat([rt_frac, node_counts], dim=-1)

        gate = torch.sigmoid(self.gate_param)
        shortcut_logits = self.shortcut_head(shortcuts)
        gnn_logits = self.gnn_head(central_x)
        logits = (1 - gate) * shortcut_logits + gate * gnn_logits
        return logits

In [0]:
import os
from tqdm.auto import tqdm

def soft_f1_loss(logits, labels, eps=1e-8):
    probs = F.softmax(logits, dim=-1)[:, 1]
    tp = (probs * labels).sum()
    fp = (probs * (1 - labels)).sum()
    fn = ((1 - probs) * labels).sum()
    f1 = (2 * tp) / (2 * tp + fp + fn + eps)
    return 1 - f1


def combined_loss(logits, labels, class_weights, epoch, warmup_epochs=10):
    ce = F.cross_entropy(logits, labels.long(), weight=class_weights.to(logits.device))
    if epoch <= warmup_epochs:
        return ce
    sf1 = soft_f1_loss(logits, labels)
    return 0.5 * ce + 0.5 * sf1


@torch.no_grad()
def evaluate(model, loader, device, class_weights=None, epoch=None):
    """Evaluate model on loader. Returns (f1, preds, labels, avg_loss).
    If class_weights and epoch are provided, also computes average loss (single pass).
    """
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    compute_loss = class_weights is not None and epoch is not None
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        preds = logits.argmax(dim=-1)
        all_preds.append(preds.cpu())
        all_labels.append(batch.y.cpu())
        if compute_loss:
            total_loss += combined_loss(logits, batch.y.float(), class_weights, epoch).item()
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    f1 = f1_score(all_labels, all_preds)
    avg_loss = total_loss / len(loader) if compute_loss else None
    return f1, all_preds, all_labels, avg_loss


CHECKPOINT_PATH = f"{EXPERIMENT_DIR}/training_checkpoint.pt"
BEST_MODEL_PATH = f"{EXPERIMENT_DIR}/best_retweet_gnn_general.pt"

In [0]:
CHECKPOINT_PATH

In [0]:
def train_model(model, raw_train_samples, raw_val_samples, epochs=50, batch_size=128,
                lr=1e-2, device="cuda", log_every_n_steps=100, patience=15,
                resume=False):
    y = np.array([s["label"] for s in raw_train_samples])
    class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    train_ds = RetweetDataset(raw_train_samples)
    val_ds = RetweetDataset(raw_val_samples)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=5e-3  # strong L2 to prevent weight specialization to train users
    )
    # LR warmup (5 epochs linear ramp) + cosine decay — slows early memorization
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=5
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs - 5))
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[5]
    )

    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * epochs
    print(f"Training on {device} | {len(train_ds)} train / {len(val_ds)} val samples")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Steps/epoch: {steps_per_epoch} | Total steps: {total_steps} | Logging every {log_every_n_steps} steps")

    best_val_f1 = 0
    global_step = 0
    running_loss = 0.0
    running_steps = 0
    start_epoch = 1
    # Classic early stopping: stop after `patience` checkpoints without val F1 improvement
    steps_since_improvement = 0

    # Training history for plotting
    HISTORY_PATH = f"{EXPERIMENT_DIR}/training_history.pkl"
    history = {
        "step": [],        # global step at each checkpoint
        "train_loss": [],  # running avg train loss at checkpoint
        "val_loss": [],    # val loss at checkpoint
        "val_f1": [],      # val f1 at checkpoint
        "epoch_step": [],  # global step at end of epoch
        "epoch_train_f1": [],  # train f1 at end of epoch
        "epoch_val_f1": [],    # val f1 at end of epoch
        "epoch_val_loss": [],  # val loss at end of epoch
    }

    # Resume from checkpoint if available
    if resume and os.path.exists(CHECKPOINT_PATH):
        print(f"  Resuming from checkpoint: {CHECKPOINT_PATH}")
        ckpt = torch.load(CHECKPOINT_PATH, weights_only=False, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt["global_step"]
        best_val_f1 = ckpt["best_val_f1"]
        steps_since_improvement = ckpt.get("steps_since_improvement", 0)
        # Restore history from previous run
        if os.path.exists(HISTORY_PATH):
            import pickle
            with open(HISTORY_PATH, "rb") as f:
                history = pickle.load(f)
            print(f"  Restored training history ({len(history['step'])} checkpoints, {len(history['epoch_step'])} epochs)")
        print(f"  Resumed at epoch {start_epoch}, global_step {global_step}, best_val_f1 {best_val_f1:.4f}")
    elif resume:
        print(f"  No checkpoint found at {CHECKPOINT_PATH}, starting from scratch.")

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        pbar = tqdm(enumerate(train_loader, 1), total=steps_per_epoch,
                    desc=f"Epoch {epoch}/{epochs}", leave=False,
                    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")

        for step_in_epoch, batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch)
            loss = combined_loss(logits, batch.y.float(), class_weights, epoch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            global_step += 1
            running_loss += loss.item()
            running_steps += 1

            pbar.set_postfix(loss=f"{loss.item():.4f}", step=global_step)

            # Log every N steps
            if global_step % log_every_n_steps == 0:
                pbar.refresh()
                avg_loss = running_loss / running_steps
                print(f"\n  --- Checkpoint at step {global_step} (epoch {epoch}/{epochs}, step {step_in_epoch}/{steps_per_epoch}) ---")

                print(f"    Computing val F1 + val loss (single pass)...")
                t_val = time.time()
                val_f1, _, _, val_loss = evaluate(model, val_loader, device,
                                                  class_weights=class_weights, epoch=epoch)
                print(f"    Val eval took {time.time() - t_val:.1f}s")
                gate_val = torch.sigmoid(model.gate_param).item()

                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    steps_since_improvement = 0
                    torch.save(model.state_dict(), BEST_MODEL_PATH)
                else:
                    steps_since_improvement += 1

                print(f"    Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f} | "
                      f"Val F1: {val_f1:.4f} | Best: {best_val_f1:.4f} | Gate: {gate_val:.4f} | "
                      f"No improvement: {steps_since_improvement}/{patience}")

                # Record history
                history["step"].append(global_step)
                history["train_loss"].append(avg_loss)
                history["val_loss"].append(val_loss)
                history["val_f1"].append(val_f1)

                # Classic early stopping: no val F1 improvement for `patience` checkpoints
                if patience and steps_since_improvement >= patience:
                    print(f"\n  Early stopping: no val F1 improvement for {patience} consecutive "
                          f"checkpoints. Best val F1: {best_val_f1:.4f}")
                    pbar.close()
                    # Save history and checkpoint before returning
                    import pickle
                    with open(HISTORY_PATH, "wb") as f:
                        pickle.dump(history, f)
                    ckpt = {
                        "epoch": epoch,
                        "global_step": global_step,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "best_val_f1": best_val_f1,
                        "steps_since_improvement": steps_since_improvement,
                    }
                    torch.save(ckpt, CHECKPOINT_PATH)
                    print(f"    Checkpoint and history saved (epoch {epoch}, step {global_step})")
                    model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True))
                    return model, history

                running_loss = 0.0
                running_steps = 0
                model.train()  # back to train mode after eval

        pbar.close()
        scheduler.step()

        # End-of-epoch: compute train F1 + val F1 (with loss)
        print(f"\n  === End of epoch {epoch}/{epochs} ===")
        print(f"    Computing train F1...")
        t_train_f1 = time.time()
        train_f1, _, _, _ = evaluate(model, train_loader, device)
        print(f"    Train F1 took {time.time() - t_train_f1:.1f}s")

        print(f"    Computing val F1 + val loss (single pass)...")
        t_val = time.time()
        val_f1, _, _, val_loss = evaluate(model, val_loader, device,
                                          class_weights=class_weights, epoch=epoch)
        print(f"    Val eval took {time.time() - t_val:.1f}s")

        gate_val = torch.sigmoid(model.gate_param).item()
        print(f"    Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Val Loss: {val_loss:.4f} | "
              f"Best: {best_val_f1:.4f} | Gate: {gate_val:.4f}")

        # Record end-of-epoch history
        history["epoch_step"].append(global_step)
        history["epoch_train_f1"].append(train_f1)
        history["epoch_val_f1"].append(val_f1)
        history["epoch_val_loss"].append(val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), BEST_MODEL_PATH)

        # Save checkpoint for resumability
        ckpt = {
            "epoch": epoch,
            "global_step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_f1": best_val_f1,
            "steps_since_improvement": steps_since_improvement,
        }
        torch.save(ckpt, CHECKPOINT_PATH)
        # Save history alongside checkpoint
        import pickle
        with open(HISTORY_PATH, "wb") as f:
            pickle.dump(history, f)
        print(f"    Checkpoint saved (epoch {epoch}, step {global_step}, best_val_f1 {best_val_f1:.4f})")

    print(f"\nTraining complete. Best val F1: {best_val_f1:.4f} | Total steps: {global_step}")
    model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True))
    return model, history

In [0]:
import pyarrow.parquet as pq
import s3fs as _s3fs
import gc as _gc

S3_GNN_BUCKET = "pablocelayes-test"
S3_GNN_PREFIX = f"learning/sna-classifier-gnn/{FINAL_TAG}/gnn_samples"
S3_GNN_PATH = f"s3://{S3_GNN_BUCKET}/{S3_GNN_PREFIX}"

def _compact_sample(s):
    """Convert sample lists to numpy arrays for ~10x memory reduction.
    RetweetDataset.get() handles numpy arrays identically to Python lists:
      - torch.tensor(np_array).t() works the same as torch.tensor(list_of_tuples).t()
      - len(np_array) returns N (correct emptiness check)
      - list(np_array) and set(np_array) work for neighbor/retweeted lookups
    """
    ei = s["edge_index"]
    if ei:
        edge_arr = np.array(ei, dtype=np.int32) if not isinstance(ei, np.ndarray) else ei
    else:
        edge_arr = np.empty((0, 2), dtype=np.int32)
    return {
        "central_user_id": int(s["central_user_id"]),
        "neighbor_ids": np.asarray(s["neighbor_ids"], dtype=np.int64),
        "retweeted_ids": np.asarray(s["retweeted_ids"], dtype=np.int64),
        "edge_index": edge_arr,
        "label": int(s["label"]),
    }

def _load_split_from_s3(split_name, fs):
    """Read a split directory of parquet files into compact sample dicts, file-by-file."""
    split_dir = f"{S3_GNN_BUCKET}/{S3_GNN_PREFIX}/split={split_name}"
    files = sorted(fs.ls(split_dir, detail=False))
    files = [f for f in files if f.endswith('.parquet')]
    samples = []
    for fi, fpath in enumerate(files):
        table = pq.read_table(fpath, filesystem=fs)
        central_ids = table.column('central_user_id').to_pylist()
        neighbor_ids = table.column('neighbor_ids').to_pylist()
        retweeted_ids = table.column('retweeted_ids').to_pylist()
        edge_srcs = table.column('edge_src').to_pylist()
        edge_dsts = table.column('edge_dst').to_pylist()
        labels = table.column('label').to_pylist()
        del table
        for i in range(len(central_ids)):
            src, dst = edge_srcs[i], edge_dsts[i]
            if src:
                edge_arr = np.column_stack([src, dst]).astype(np.int32)
            else:
                edge_arr = np.empty((0, 2), dtype=np.int32)
            samples.append({
                "central_user_id": central_ids[i],
                "neighbor_ids": np.array(neighbor_ids[i], dtype=np.int64),
                "retweeted_ids": np.array(retweeted_ids[i], dtype=np.int64),
                "edge_index": edge_arr,
                "label": labels[i],
            })
        del central_ids, neighbor_ids, retweeted_ids, edge_srcs, edge_dsts, labels
        if (fi + 1) % 5 == 0:
            _gc.collect()
            print(f"    {split_name}: {fi+1}/{len(files)} files loaded ({len(samples)} samples)")
    return samples

try:
    # Try loading from S3 parquet cache (PyArrow direct, no Spark)
    _fs = _s3fs.S3FileSystem()
    train_dir = f"{S3_GNN_BUCKET}/{S3_GNN_PREFIX}/split=train"
    if not _fs.exists(train_dir):
        raise FileNotFoundError(f"No cache at {train_dir}")

    # Free large objects no longer needed when loading from cache
    for _v in ['user_data', 'graph', 'baseline_f1s', 'all_baseline_test_preds']:
        if _v in globals():
            del globals()[_v]
    _gc.collect()

    print(f"Loading GNN samples from {S3_GNN_PATH}...")
    all_train_samples = _load_split_from_s3("train", _fs)
    print(f"  Train: {len(all_train_samples)}")
    _gc.collect()
    all_test_samples = _load_split_from_s3("test", _fs)
    print(f"  Test: {len(all_test_samples)}")

    # Reconstruct user_test_samples from test samples keyed by central_user_id
    user_test_samples = {}
    for s in all_test_samples:
        uid = s["central_user_id"]
        user_test_samples.setdefault(uid, []).append(s)

    gnn_failed_users = []
    print(f"  Users with test samples: {len(user_test_samples)}")
    del _fs
except Exception as _cache_err:
    print(f"No S3 cache found ({_cache_err}), building from scratch...")

    # ---------------------------------------------------------------
    # Build GNN samples with group-based train/test split:
    #   TRAIN: train data from TRAIN_GROUPS (u_train, au_train)
    #   TEST:  test data from TRAIN_GROUPS + ALL data from TEST_GROUPS
    # ---------------------------------------------------------------
    all_train_samples = []
    all_test_samples = []
    user_test_samples = {}  # uid -> list of test samples (for per-user eval later)
    gnn_failed_users = []

    processing_plan = []
    for group in TRAIN_GROUPS:
        for uid in user_data.get(group, {}):
            processing_plan.append((group, uid, "train_group"))
    for group in TEST_GROUPS:
        for uid in user_data.get(group, {}):
            processing_plan.append((group, uid, "test_group"))

    print(f"Processing {len(processing_plan)} users ({sum(1 for _,_,r in processing_plan if r=='train_group')} train-group, "
          f"{sum(1 for _,_,r in processing_plan if r=='test_group')} test-group)")

    t0 = time.time()
    for i, (group, uid, role) in enumerate(processing_plan):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data[group][uid]
        try:
            train_samples, test_samples = create_gnn_train_val_samples(uid, graph, X_tr, y_tr, X_te, y_te)
            # Compact samples to numpy for memory efficiency
            train_samples = [_compact_sample(s) for s in train_samples]
            test_samples = [_compact_sample(s) for s in test_samples]

            if role == "train_group":
                all_train_samples.extend(train_samples)
                all_test_samples.extend(test_samples)
                user_test_samples[uid] = test_samples
            else:
                all_test_samples.extend(train_samples)
                all_test_samples.extend(test_samples)
                user_test_samples[uid] = train_samples + test_samples

        except Exception as e:
            gnn_failed_users.append((group, uid, str(e)))
            print(f"  Warning: Failed to transform user {uid} ({group}): {e}")
            continue

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(processing_plan) - i - 1)
        n_tr = len(train_samples) if role == "train_group" else 0
        n_te = len(test_samples) if role == "train_group" else len(train_samples) + len(test_samples)
        print(f"  [{i+1:>3}/{len(processing_plan)}] {group}/{uid}  "
              f"+{n_tr} train / +{n_te} test  "
              f"({user_time:.1f}s | elapsed {elapsed:.0f}s | ETA {remaining:.0f}s)")

    total_time = time.time() - t0
    print(f"\nGNN dataset ready in {total_time:.1f}s ({total_time/len(processing_plan):.1f}s/user avg):")
    print(f"  Total train samples: {len(all_train_samples)}")
    print(f"  Total test samples:  {len(all_test_samples)}")
    print(f"  Users with test samples: {len(user_test_samples)}")
    print(f"  Failed users: {len(gnn_failed_users)}")


# Shuffle train samples so batches mix users
from random import shuffle as shuffle_list
shuffle_list(all_train_samples)

In [0]:
# Save GNN samples to S3 as parquet — using PyArrow directly (no Spark/JVM overhead)
# Spark's createDataFrame serializes Python->JVM which doubles memory and killed the driver.
import gc
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs

# Free objects no longer needed — reclaim memory before the write
for _var in ['user_data', 'graph', 'processing_plan', 'user_test_samples']:
    if _var in dir():
        exec(f'del {_var}')
gc.collect()

S3_GNN_BUCKET = "pablocelayes-test"
S3_GNN_PREFIX = f"learning/sna-classifier-gnn/{FINAL_TAG}/gnn_samples"
S3_GNN_PATH = f"s3://{S3_GNN_BUCKET}/{S3_GNN_PREFIX}"  # for Spark reads later
CHUNK_SIZE = 10_000  # rows per parquet file — small to control peak memory

# PyArrow schema with nested list columns
arrow_schema = pa.schema([
    ('central_user_id', pa.int64()),
    ('neighbor_ids', pa.list_(pa.int64())),
    ('retweeted_ids', pa.list_(pa.int64())),
    ('edge_src', pa.list_(pa.int64())),
    ('edge_dst', pa.list_(pa.int64())),
    ('label', pa.int32()),
])

def samples_to_arrow_table(samples):
    """Convert a batch of sample dicts to a PyArrow Table, column-by-column to minimize peak memory."""
    n = len(samples)
    # Build and convert one column at a time to avoid holding all columns as Python lists simultaneously
    col_central = pa.array([int(s["central_user_id"]) for s in samples], type=pa.int64())
    col_label = pa.array([int(s["label"]) for s in samples], type=pa.int32())

    col_neighbors = pa.array([s["neighbor_ids"] for s in samples], type=pa.list_(pa.int64()))
    col_retweeted = pa.array([s["retweeted_ids"] for s in samples], type=pa.list_(pa.int64()))

    edge_src_data = []
    edge_dst_data = []
    for s in samples:
        ei = s["edge_index"]
        if ei:
            edge_src_data.append([e[0] for e in ei])
            edge_dst_data.append([e[1] for e in ei])
        else:
            edge_src_data.append([])
            edge_dst_data.append([])
    col_edge_src = pa.array(edge_src_data, type=pa.list_(pa.int64()))
    del edge_src_data
    col_edge_dst = pa.array(edge_dst_data, type=pa.list_(pa.int64()))
    del edge_dst_data

    return pa.table({
        'central_user_id': col_central,
        'neighbor_ids': col_neighbors,
        'retweeted_ids': col_retweeted,
        'edge_src': col_edge_src,
        'edge_dst': col_edge_dst,
        'label': col_label,
    })

def write_split(samples, split_name, fs):
    """Write a split as chunked parquet files directly to S3."""
    split_dir = f"{S3_GNN_BUCKET}/{S3_GNN_PREFIX}/split={split_name}"
    # Clear existing split directory
    if fs.exists(split_dir):
        fs.rm(split_dir, recursive=True)
    fs.mkdirs(split_dir, exist_ok=True)

    n_chunks = (len(samples) + CHUNK_SIZE - 1) // CHUNK_SIZE
    for chunk_idx in range(n_chunks):
        start = chunk_idx * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, len(samples))
        table = samples_to_arrow_table(samples[start:end])
        out_path = f"{split_dir}/part-{chunk_idx:05d}.snappy.parquet"
        with fs.open(out_path, 'wb') as f:
            pq.write_table(table, f, compression='snappy')
        del table
        gc.collect()
        print(f"    {split_name} part {chunk_idx+1}/{n_chunks} written ({end - start} rows)")

fs = s3fs.S3FileSystem()
print(f"Writing {len(all_train_samples)} train + {len(all_test_samples)} test samples to S3...")
print(f"  Path: {S3_GNN_PATH}")
print(f"  Chunk size: {CHUNK_SIZE:,} rows/file")

write_split(all_train_samples, "train", fs)
write_split(all_test_samples, "test", fs)
print(f"\n  Done! Saved to {S3_GNN_PATH}")

# Save failed users
if gnn_failed_users:
    failed_table = pa.table({
        'group': pa.array([str(g) for g, _, _ in gnn_failed_users]),
        'user_id': pa.array([str(u) for _, u, _ in gnn_failed_users]),
        'error': pa.array([str(e) for _, _, e in gnn_failed_users]),
    })
    failed_path = f"{S3_GNN_BUCKET}/{S3_GNN_PREFIX}_failed_users/part-00000.parquet"
    fs.mkdirs(f"{S3_GNN_BUCKET}/{S3_GNN_PREFIX}_failed_users", exist_ok=True)
    with fs.open(failed_path, 'wb') as f:
        pq.write_table(failed_table, f)
    print(f"  Saved {len(gnn_failed_users)} failed users")
else:
    print("  No failed users to save.")

In [0]:
import gc
gc.collect()
torch.cuda.empty_cache()

# Validation from test set (to detect overfitting to train distribution)
# Sample a fixed subset of test set as val — same size as before (~3500)
from random import Random
VAL_SIZE = 20000
val_rng = Random(42)  # fixed seed for reproducibility across runs
val_indices = val_rng.sample(range(len(all_test_samples)), min(VAL_SIZE, len(all_test_samples)))
val_indices_set = set(val_indices)
val_samples_final = [all_test_samples[i] for i in val_indices]

# All train samples used for training (no holdout from train)
train_samples_final = all_train_samples

print(f"Train: {len(train_samples_final)} samples (full train set)")
print(f"Val: {len(val_samples_final)} samples (sampled from test set, fixed seed)")
print(f"Test set (full, for final eval): {len(all_test_samples)} samples")

In [0]:
# Anti-overfitting: strong regularization to prevent memorizing train users' graph patterns.
# Test set has entirely unseen users — model must generalize graph structure, not memorize it.
model = RetweetGNN(
    ff_hidden_dim=64,
    gcn_hidden_dim=64,
    transformer_dim=64,
    transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH,
    device=device,
    dropout=0.5,           # high dropout forces redundant representations that transfer
    drop_edge_rate=0.4,    # aggressively drop edges — prevents memorizing specific neighbor patterns
).to(device)

# Gate at 0.0 → sigmoid=0.5 (balanced start). Let the model earn GNN contribution.
# Shortcut head uses aggregate stats (rt_frac, node_count) which generalize better to unseen users.
# If GNN can't beat shortcut on val, gate will stay low — that's fine.
with torch.no_grad():
    model.gate_param.fill_(0.0)

In [0]:
# Classic early stopping: stop after this many checkpoints without val F1 improvement.
# Set to None to disable early stopping and train for all epochs.
PATIENCE = 20  # at log_every_n_steps=300, this means ~6000 steps without improvement

In [0]:
# Train on combined shuffled data — anti-overfitting configuration:
#   - lr=3e-3 with 5-epoch linear warmup: prevents fast memorization in early steps
#   - weight_decay=5e-3: strong L2 prevents weight specialization
#   - dropout=0.5 + drop_edge=0.4: heavy stochastic regularization
#   - gate_param=0.0 (50/50): shortcut generalizes better; GNN must earn its contribution
model, history = train_model(
    model=model,
    raw_train_samples=train_samples_final,
    raw_val_samples=val_samples_final,
    epochs=60,
    batch_size=256,
    device=device,
    lr=3e-3,
    log_every_n_steps=300,
    patience=PATIENCE,
    resume=False,  # fresh start — new hyperparameters
)